# Chapter 14 Walkthrough: Professional Engineering with VS Code, Virtual Environments, Modules, and Packages

**Book Reference:** Chapter 14, Sections 14.1 – 14.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every filesystem demo runs inside `tempfile.TemporaryDirectory()` so zero stray folders or files remain in your workspace.

## Section 14.3.1 — Notebook-to-Module Refactoring & `if __name__ == "__main__":`

In [ ]:
import importlib.util
from pathlib import Path
import tempfile

module_code = '''
def classify_hb(hb_g_dl, cutoff=11.0):
    """Return 'Anemic' if hb_g_dl < cutoff else 'Normal'."""
    return "Anemic" if hb_g_dl < cutoff else "Normal"

if __name__ == "__main__":
    print("CLI smoke test:", classify_hb(10.2))
'''

with tempfile.TemporaryDirectory() as tmp:
    mod_path = Path(tmp) / "triage_utils.py"
    mod_path.write_text(module_code.strip() + "\n", encoding="utf-8")

    spec = importlib.util.spec_from_file_location(
        "triage_utils", mod_path
    )
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    print("Imported __name__:", mod.__name__)
    print("Hb 10.4 ->", mod.classify_hb(10.4))
    print("Hb 12.1 ->", mod.classify_hb(12.1))


## Section 14.3.2 — Virtual Environments, `sys.prefix`, & Dependency Files

In [ ]:
from pathlib import Path
import sys
import tempfile

in_venv = sys.prefix != sys.base_prefix
print("Python version:", sys.version.split()[0])
print("Inside virtual env:", in_venv)

req_text = (
    "# Pinned cohort pipeline dependencies\n"
    "pandas==2.2.0\n"
    "numpy>=1.26\n"
    "scipy\n"
)

with tempfile.TemporaryDirectory() as tmp:
    req_path = Path(tmp) / "requirements.txt"
    req_path.write_text(req_text, encoding="utf-8")
    reqs = {}
    with open(req_path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            for op in ("==", ">=", "<=", "~="):
                if op in line:
                    pkg, ver = line.split(op, 1)
                    reqs[pkg.strip().lower()] = op + ver.strip()
                    break
            else:
                reqs[line.lower()] = ""
    print("Parsed requirements:", reqs)


## Section 14.3.3 — Standard `src/` Package Layout & `.env` Configuration

In [ ]:
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    (root / "src" / "mch_tools").mkdir(parents=True)
    (root / "tests").mkdir()
    (root / "pyproject.toml").write_text(
        '[project]\nname = "mch_tools"\nversion = "0.1.0"\n',
        encoding="utf-8",
    )
    (root / "src" / "mch_tools" / "__init__.py").write_text(
        '__version__ = "0.1.0"\n', encoding="utf-8"
    )
    env_path = root / ".env"
    env_path.write_text(
        '# Local secrets (never commit to Git!)\n'
        'STUDY_ID="MCH-BD-2026"\n'
        "API_TOKEN='demo-key-99'\n",
        encoding="utf-8",
    )

    has_pyp = (root / "pyproject.toml").is_file()
    has_init = (root / "src" / "mch_tools" / "__init__.py").is_file()
    has_tests = (root / "tests").is_dir()
    print("Layout valid:", has_pyp and has_init and has_tests)

    env_vars = {}
    for raw in env_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        key, val = line.split("=", 1)
        val = val.strip()
        if len(val) >= 2 and val[0] == val[-1] and val[0] in ('"', "'"):
            val = val[1:-1]
        env_vars[key.strip()] = val
    print("Loaded .env keys:", env_vars)


## Section 14.3.4 — Reproducible CLI Tools with `argparse.parse_args([...])`

In [ ]:
import argparse

parser = argparse.ArgumentParser(
    description="Screen a cohort CSV for maternal anemia."
)
parser.add_argument("--input", type=str, required=True)
parser.add_argument("--threshold", type=float, default=11.0)
parser.add_argument("--verbose", action="store_true")

args = parser.parse_args(
    ["--input", "dhaka_mch.csv", "--threshold", "10.5", "--verbose"]
)
print("Input file:", args.input)
print("Cutoff (g/dL):", args.threshold, type(args.threshold).__name__)
print("Verbose mode:", args.verbose)


## Section 14.3.5 — Break It On Purpose! (Packaging, CLI & `.env` Error Anatomy)

In [ ]:
# Intentional Bug #1: ModuleNotFoundError — missing or uninstalled pkg
import nonexistent_clinic_pkg


In [ ]:
# Intentional Bug #2: SystemExit — missing required --input CLI flag
import argparse

parser = argparse.ArgumentParser(prog="screen_cohort")
parser.add_argument("--input", type=str, required=True)
parser.parse_args([])


In [ ]:
# Intentional Bug #3: ValueError — malformed .env line missing '='
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    bad_env = Path(tmp) / ".env"
    bad_env.write_text("API_TOKEN_WITHOUT_EQUALS\n", encoding="utf-8")
    for raw in bad_env.read_text(encoding="utf-8").splitlines():
        key, value = raw.split("=", 1)


## Section 14.4 — Worked Problem-Structuring Blueprint: Reproducible Package Readiness Check

In [ ]:
from pathlib import Path
import re
import tempfile

def check_package_readiness(root_dir, raw_pkg_name, req_path):
    """Normalize name, check src/ layout, and flag unpinned deps."""
    cleaned = raw_pkg_name.strip().lower()
    cleaned = cleaned.replace("-", "_").replace(" ", "_")
    pkg = re.sub(r"_+", "_", cleaned).strip("_")

    root = Path(root_dir)
    layout_ok = (
        (root / "pyproject.toml").is_file()
        and (root / "src" / pkg / "__init__.py").is_file()
        and (root / "tests").is_dir()
    )

    unpinned = []
    for raw in Path(req_path).read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        if "==" not in line:
            name = re.split(r"[><~=]", line, maxsplit=1)[0]
            unpinned.append(name.strip().lower())

    unpinned.sort()
    return {
        "package": pkg,
        "layout_ok": layout_ok,
        "unpinned": unpinned,
        "ready": layout_ok and len(unpinned) == 0,
    }

with tempfile.TemporaryDirectory() as tmp:
    proj = Path(tmp)
    (proj / "src" / "dhaka_epi").mkdir(parents=True)
    (proj / "tests").mkdir()
    (proj / "pyproject.toml").write_text(
        '[project]\nname = "dhaka_epi"\n', encoding="utf-8"
    )
    (proj / "src" / "dhaka_epi" / "__init__.py").write_text(
        '__version__ = "0.1.0"\n', encoding="utf-8"
    )
    req = proj / "requirements.txt"
    req.write_text("pandas==2.2.0\nnumpy>=1.26\n", encoding="utf-8")
    report = check_package_readiness(proj, "Dhaka-Epi", req)
    print("Readiness audit:", report)
